# Fed-Vis: Training on Google Colab

This notebook trains the Attention U-Net using data stored on Google Drive.
Everything runs on Colab — nothing downloaded to your PC.

**Before running:** Make sure you have these files in `My Drive/Capstone/`:
- `MICCAI_FeTS2022_TrainingData.zip` (12.47 GB)
- `nrrd_lung.zip` (25.2 MB)
- `Processed_data_nii.zip` (prostate)

**Runtime:** Go to `Runtime > Change runtime type > T4 GPU`

---
## 1. Setup

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Capstone'
print('Files in Capstone folder:')
for f in os.listdir(DRIVE_ROOT):
    size = os.path.getsize(os.path.join(DRIVE_ROOT, f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

In [ ]:
# clone the fed_vis repo
!git clone https://github.com/JoakBouy/Fed-Vis_Mission-Capstone.git /content/FedFMS 2>/dev/null || echo 'Already cloned'

import sys
sys.path.insert(0, '/content/FedFMS/fed_vis/src')

# install deps
!pip install -q nibabel flwr hydra-core omegaconf pynrrd SimpleITK

In [ ]:
# verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

---
## 2. Extract & Explore Datasets

We extract to `/content/data/` (Colab local disk, fast). Not back to Drive.

In [ ]:
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# --- FeTS 2022 (brain) ---
fets_zip = os.path.join(DRIVE_ROOT, 'MICCAI_FeTS2022_TrainingData.zip')
fets_dir = os.path.join(DATA_DIR, 'FeTS2022')

if not os.path.exists(fets_dir):
    print('Extracting FeTS2022 (this takes a few minutes)...')
    !unzip -q "{fets_zip}" -d "{fets_dir}"
    print('Done.')
else:
    print('FeTS2022 already extracted.')

# show structure
print('\nFeTS2022 top-level:')
!ls "{fets_dir}" | head -20

In [ ]:
# --- Prostate ---
prostate_zip = os.path.join(DRIVE_ROOT, 'Processed_data_nii.zip')
prostate_dir = os.path.join(DATA_DIR, 'Prostate')

if not os.path.exists(prostate_dir):
    print('Extracting Prostate...')
    !unzip -q "{prostate_zip}" -d "{prostate_dir}"
    print('Done.')
else:
    print('Prostate already extracted.')

print('\nProstate top-level:')
!ls "{prostate_dir}"
print('\nFirst site:')
!ls "{prostate_dir}"/*/ 2>/dev/null | head -20

In [ ]:
# --- Lung (NRRD format) ---
lung_zip = os.path.join(DRIVE_ROOT, 'nrrd_lung.zip')
lung_raw_dir = os.path.join(DATA_DIR, 'Lung_raw')

if not os.path.exists(lung_raw_dir):
    print('Extracting Lung...')
    !unzip -q "{lung_zip}" -d "{lung_raw_dir}"
    print('Done.')
else:
    print('Lung already extracted.')

print('\nLung raw top-level:')
!find "{lung_raw_dir}" -type f | head -20

---
## 3. Explore Data Formats

Let's look at one sample from each dataset to understand shapes and values.

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

# find a FeTS sample
fets_niis = sorted(glob(os.path.join(fets_dir, '**/*t1.nii.gz'), recursive=True))
if not fets_niis:
    fets_niis = sorted(glob(os.path.join(fets_dir, '**/*.nii.gz'), recursive=True))

print(f'Found {len(fets_niis)} FeTS NIfTI files')
if fets_niis:
    print(f'Example: {fets_niis[0]}')
    img = nib.load(fets_niis[0])
    vol = img.get_fdata()
    print(f'Shape: {vol.shape}')
    print(f'Dtype: {vol.dtype}')
    print(f'Range: [{vol.min():.1f}, {vol.max():.1f}]')
    print(f'Spacing: {img.header.get_zooms()}')

    # show middle slice
    mid = vol.shape[2] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(vol[:, :, mid], cmap='gray')
    plt.title(f'FeTS brain slice {mid}')
    plt.axis('off')
    plt.show()

In [ ]:
# find a prostate sample
prost_niis = sorted(glob(os.path.join(prostate_dir, '**/*.nii.gz'), recursive=True))
if not prost_niis:
    prost_niis = sorted(glob(os.path.join(prostate_dir, '**/*.nii'), recursive=True))

print(f'Found {len(prost_niis)} Prostate NIfTI files')
if prost_niis:
    print(f'Example: {prost_niis[0]}')
    img = nib.load(prost_niis[0])
    vol = img.get_fdata()
    print(f'Shape: {vol.shape}, Range: [{vol.min():.1f}, {vol.max():.1f}]')

    mid = vol.shape[2] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(vol[:, :, mid], cmap='gray')
    plt.title(f'Prostate slice {mid}')
    plt.axis('off')
    plt.show()

---
## 4. Convert Lung NRRD → NIfTI

Our model pipeline expects NIfTI format. The lung data is in NRRD,
so we convert it here.

In [ ]:
import SimpleITK as sitk

def convert_nrrd_to_nifti(nrrd_path, output_path):
    """Convert a single .nrrd file to .nii.gz"""
    img = sitk.ReadImage(nrrd_path)
    sitk.WriteImage(img, output_path)

# find all nrrd files
nrrd_files = sorted(glob(os.path.join(lung_raw_dir, '**/*.nrrd'), recursive=True))
print(f'Found {len(nrrd_files)} NRRD files')

if nrrd_files:
    # peek at the first one
    print(f'\nFirst file: {nrrd_files[0]}')
    img = sitk.ReadImage(nrrd_files[0])
    arr = sitk.GetArrayFromImage(img)
    print(f'Shape: {arr.shape}')
    print(f'Spacing: {img.GetSpacing()}')
    print(f'Range: [{arr.min():.1f}, {arr.max():.1f}]')

In [ ]:
# convert all NRRD to NIfTI
lung_nifti_dir = os.path.join(DATA_DIR, 'Lung')
os.makedirs(os.path.join(lung_nifti_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(lung_nifti_dir, 'masks'), exist_ok=True)

converted = 0
for nrrd_path in nrrd_files:
    fname = os.path.basename(nrrd_path)
    name_no_ext = os.path.splitext(fname)[0]
    
    # figure out if this is an image or mask based on filename
    lower = fname.lower()
    if 'mask' in lower or 'seg' in lower or 'label' in lower:
        out_dir = os.path.join(lung_nifti_dir, 'masks')
    else:
        out_dir = os.path.join(lung_nifti_dir, 'images')
    
    out_path = os.path.join(out_dir, name_no_ext + '.nii.gz')
    
    if not os.path.exists(out_path):
        convert_nrrd_to_nifti(nrrd_path, out_path)
        converted += 1

print(f'Converted {converted} files.')
print(f'Images: {len(os.listdir(os.path.join(lung_nifti_dir, "images")))}')
print(f'Masks:  {len(os.listdir(os.path.join(lung_nifti_dir, "masks")))}')

---
## 5. Check Data Structure

Let's see the full layout so we can point our loaders at the right paths.

In [ ]:
print('=== DATA DIRECTORY ===')
!find /content/data -maxdepth 3 -type d
print()
print('=== File counts ===')
for d in sorted(glob('/content/data/**/', recursive=True)):
    files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
    if files:
        print(f'  {d}: {len(files)} files')

---
## 6. Quick Model Test

Before training, let's verify the model runs on a real volume.

In [ ]:
from fedvis.models.attention_unet import AttentionUNet3D
from fedvis.models.losses import CombinedLoss, dice_coefficient

model = AttentionUNet3D(in_channels=1, out_channels=1, base_features=32)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters')
print(f'Device: {device}')

# test with a dummy input
dummy = torch.randn(1, 1, 64, 128, 128).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}')
print('Model works!')

In [ ]:
# now try on a REAL FeTS volume
if fets_niis:
    from fedvis.data.preprocessing import normalize_intensity, crop_or_pad_volume
    
    raw_vol = nib.load(fets_niis[0]).get_fdata().astype(np.float32)
    print(f'Raw shape: {raw_vol.shape}')
    
    # preprocess: normalize + resize to our target
    # z-score on non-zero voxels
    mask = raw_vol > 0
    if mask.sum() > 0:
        raw_vol[mask] = (raw_vol[mask] - raw_vol[mask].mean()) / (raw_vol[mask].std() + 1e-8)
    raw_vol[~mask] = 0
    
    # crop/pad to 64x128x128
    from scipy.ndimage import zoom
    target = (64, 128, 128)
    factors = [t / s for t, s in zip(target, raw_vol.shape)]
    vol_resized = zoom(raw_vol, factors, order=1)
    print(f'Resized: {vol_resized.shape}')
    
    # run through model
    tensor = torch.from_numpy(vol_resized).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor))
    
    pred_np = pred[0, 0].cpu().numpy()
    print(f'Prediction range: [{pred_np.min():.3f}, {pred_np.max():.3f}]')
    print(f'Predicted tumor %: {(pred_np > 0.5).sum() / pred_np.size * 100:.1f}%')
    print('(With random weights, this is meaningless — just checking the pipeline works)')
    
    # visualize
    mid = 32
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(vol_resized[mid], cmap='gray')
    axes[0].set_title('Input (preprocessed)')
    axes[1].imshow(pred_np[mid], cmap='hot', vmin=0, vmax=1)
    axes[1].set_title('Prediction (untrained)')
    axes[2].imshow(vol_resized[mid], cmap='gray')
    axes[2].imshow(pred_np[mid], cmap='Reds', alpha=0.3)
    axes[2].set_title('Overlay')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 7. Prepare DataLoaders

Now we build proper DataLoaders that match the actual data structure.
We'll adapt based on what we found in step 5.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
import random

TARGET_SHAPE = (64, 128, 128)

class SimpleNiftiDataset(Dataset):
    """Loads NIfTI volumes and masks. Works with any folder structure.
    
    Pairs image files with mask files by matching filenames.
    Handles resizing to a standard shape on the fly.
    """
    def __init__(self, image_paths, mask_paths, target_shape=TARGET_SHAPE, normalize='zscore'):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)
        self.target_shape = target_shape
        self.normalize = normalize
        
        assert len(self.image_paths) == len(self.mask_paths), \
            f'Mismatch: {len(self.image_paths)} images vs {len(self.mask_paths)} masks'
        print(f'Dataset: {len(self)} samples, normalize={normalize}')
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # load
        vol = nib.load(self.image_paths[idx]).get_fdata().astype(np.float32)
        mask = nib.load(self.mask_paths[idx]).get_fdata().astype(np.float32)
        
        # if 4D (multi-modal), take first modality
        if vol.ndim == 4:
            vol = vol[..., 0]
        if mask.ndim == 4:
            mask = mask[..., 0]
        
        # normalize
        if self.normalize == 'zscore':
            fg = vol > 0
            if fg.sum() > 0:
                vol[fg] = (vol[fg] - vol[fg].mean()) / (vol[fg].std() + 1e-8)
            vol[~fg] = 0
        elif self.normalize == 'minmax':
            vmin, vmax = vol.min(), vol.max()
            if vmax > vmin:
                vol = (vol - vmin) / (vmax - vmin)
        
        # binarize mask
        mask = (mask > 0).astype(np.float32)
        
        # resize to target shape
        if vol.shape != self.target_shape:
            factors = [t / s for t, s in zip(self.target_shape, vol.shape)]
            vol = zoom(vol, factors, order=1)       # bilinear for images
            mask = zoom(mask, factors, order=0)     # nearest for masks
        
        # to tensor: add channel dim
        vol_t = torch.from_numpy(vol).unsqueeze(0)     # (1, D, H, W)
        mask_t = torch.from_numpy(mask).unsqueeze(0)   # (1, D, H, W)
        
        return vol_t, mask_t

In [ ]:
def find_paired_files(data_dir):
    """Find image/mask pairs in common FeTS/Prostate/Lung folder structures.
    
    Looks for common patterns:
    - images/ + labels/ (FeTS style)
    - images/ + masks/ (lung style)
    - *_t1.nii.gz + *_seg.nii.gz (BraTS naming)
    """
    images, masks = [], []
    
    # pattern 1: FeTS/BraTS naming (same dir, *_t1.nii.gz + *_seg.nii.gz)
    segs = sorted(glob(os.path.join(data_dir, '**/*seg*.nii.gz'), recursive=True))
    if segs:
        for seg_path in segs:
            # find matching t1
            base_dir = os.path.dirname(seg_path)
            patient = os.path.basename(base_dir)
            t1_candidates = glob(os.path.join(base_dir, '*t1.nii.gz'))
            if not t1_candidates:
                t1_candidates = glob(os.path.join(base_dir, '*T1.nii.gz'))
            if not t1_candidates:
                # try flair or t2
                t1_candidates = glob(os.path.join(base_dir, '*flair*.nii.gz'))
            if not t1_candidates:
                t1_candidates = glob(os.path.join(base_dir, '*.nii.gz'))
                t1_candidates = [c for c in t1_candidates if 'seg' not in c.lower()]
            
            if t1_candidates:
                images.append(t1_candidates[0])
                masks.append(seg_path)
        if images:
            print(f'Found {len(images)} BraTS-style pairs')
            return images, masks
    
    # pattern 2: images/ + labels/ or masks/ folders
    img_dir = None
    mask_dir = None
    for name in ['images', 'image', 'img']:
        d = os.path.join(data_dir, name)
        if os.path.isdir(d):
            img_dir = d
            break
    for name in ['labels', 'masks', 'label', 'mask', 'seg']:
        d = os.path.join(data_dir, name)
        if os.path.isdir(d):
            mask_dir = d
            break
    
    if img_dir and mask_dir:
        img_files = sorted(glob(os.path.join(img_dir, '*.nii*')))
        for img_path in img_files:
            fname = os.path.basename(img_path)
            mask_path = os.path.join(mask_dir, fname)
            if os.path.exists(mask_path):
                images.append(img_path)
                masks.append(mask_path)
        print(f'Found {len(images)} folder-style pairs in {data_dir}')
        return images, masks
    
    print(f'WARNING: Could not find pairs in {data_dir}')
    print(f'Contents: {os.listdir(data_dir)[:10]}')
    return [], []

# test on each dataset
print('--- FeTS ---')
fets_imgs, fets_masks = find_paired_files(fets_dir)
if fets_imgs:
    print(f'  Example image: {fets_imgs[0]}')
    print(f'  Example mask:  {fets_masks[0]}')

print('\n--- Prostate ---')
prost_imgs, prost_masks = find_paired_files(prostate_dir)

print('\n--- Lung ---')
lung_imgs, lung_masks = find_paired_files(lung_nifti_dir)

---
## 8. Train on FeTS (Brain) — Local Baseline

Train for ~30 epochs on the brain dataset first.

In [ ]:
# split into train/val
random.seed(42)
n = len(fets_imgs)
indices = list(range(n))
random.shuffle(indices)

split = int(0.8 * n)
train_idx = indices[:split]
val_idx = indices[split:]

# use a subset if dataset is huge (speeds up training)
MAX_TRAIN = 50   # adjust based on time available
MAX_VAL = 10
train_idx = train_idx[:MAX_TRAIN]
val_idx = val_idx[:MAX_VAL]

train_ds = SimpleNiftiDataset(
    [fets_imgs[i] for i in train_idx],
    [fets_masks[i] for i in train_idx],
    normalize='zscore',
)
val_ds = SimpleNiftiDataset(
    [fets_imgs[i] for i in val_idx],
    [fets_masks[i] for i in val_idx],
    normalize='zscore',
)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} samples, Val: {len(val_ds)} samples')

In [ ]:
# sanity check: look at a batch
vol_batch, mask_batch = next(iter(train_loader))
print(f'Volume batch: {vol_batch.shape}, dtype={vol_batch.dtype}')
print(f'Mask batch:   {mask_batch.shape}, unique={torch.unique(mask_batch).tolist()}')

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
mid = 32
axes[0, 0].imshow(vol_batch[0, 0, mid].numpy(), cmap='gray')
axes[0, 0].set_title('Sample 1 - MRI')
axes[0, 1].imshow(mask_batch[0, 0, mid].numpy(), cmap='Reds')
axes[0, 1].set_title('Sample 1 - Tumor mask')
axes[1, 0].imshow(vol_batch[1, 0, mid].numpy(), cmap='gray')
axes[1, 0].set_title('Sample 2 - MRI')
axes[1, 1].imshow(mask_batch[1, 0, mid].numpy(), cmap='Reds')
axes[1, 1].set_title('Sample 2 - Tumor mask')
for ax in axes.flat: ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from fedvis.training.trainer import Trainer

# fresh model
model = AttentionUNet3D(in_channels=1, out_channels=1, base_features=32)

OUTPUT_DIR = '/content/outputs/fets_local'
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg={'lr': 1e-4, 'epochs': 30, 'dice_weight': 1.0, 'bce_weight': 1.0},
    output_dir=OUTPUT_DIR,
    device=device,
)

print('Starting training...')
best_dice = trainer.train(num_epochs=30)
print(f'\nBest validation dice: {best_dice:.4f}')

---
## 9. Visualize Results

In [ ]:
# load best model
ckpt = torch.load(os.path.join(OUTPUT_DIR, 'checkpoints/best.pth'), map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()

# predict on validation samples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
titles = ['Input', 'Ground Truth', 'Prediction']

with torch.no_grad():
    for row in range(3):
        if row >= len(val_ds):
            break
        vol, mask = val_ds[row]
        vol = vol.unsqueeze(0).to(device)
        
        pred = torch.sigmoid(model(vol))
        pred_np = pred[0, 0].cpu().numpy()
        vol_np = vol[0, 0].cpu().numpy()
        mask_np = mask[0].numpy()
        
        mid = 32
        d = dice_coefficient(
            pred.cpu(), mask.unsqueeze(0).float()
        ).item()
        
        axes[row, 0].imshow(vol_np[mid], cmap='gray')
        axes[row, 1].imshow(mask_np[mid], cmap='Reds', vmin=0, vmax=1)
        axes[row, 2].imshow(pred_np[mid], cmap='Reds', vmin=0, vmax=1)
        axes[row, 2].set_title(f'Dice: {d:.3f}')
        
        if row == 0:
            for c, t in enumerate(titles):
                axes[0, c].set_title(t)

for ax in axes.flat: ax.axis('off')
plt.suptitle('Local Training Results (FeTS Brain)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 10. Attention Maps — Where the Model Looks

In [ ]:
# get attention maps
vol, mask = val_ds[0]
vol = vol.unsqueeze(0).to(device)

with torch.no_grad():
    pred, attn_maps = model(vol, return_attention=True)

print(f'Got {len(attn_maps)} attention maps:')
for i, a in enumerate(attn_maps):
    print(f'  Level {i}: {a.shape}')

# visualize attention at each level
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
mid = 32

vol_np = vol[0, 0].cpu().numpy()
axes[0].imshow(vol_np[mid], cmap='gray')
axes[0].set_title('Input')

for i, a in enumerate(attn_maps):
    attn_np = a[0, 0].cpu().numpy()
    # resize attention to input size for overlay
    attn_mid = attn_np.shape[0] // 2
    attn_slice = zoom(attn_np[attn_mid], 
                       [128 / attn_np.shape[1], 128 / attn_np.shape[2]], order=1)
    axes[i+1].imshow(vol_np[mid], cmap='gray')
    axes[i+1].imshow(attn_slice, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    axes[i+1].set_title(f'Attention L{i} ({attn_np.shape[1]}x{attn_np.shape[2]})')

for ax in axes: ax.axis('off')
plt.suptitle('Attention Maps — Where the Model Focuses', fontsize=14)
plt.tight_layout()
plt.show()

---
## 11. Save Model to Google Drive

Copy the trained checkpoint back to Drive so it persists.

In [ ]:
import shutil

save_dir = os.path.join(DRIVE_ROOT, 'trained_models')
os.makedirs(save_dir, exist_ok=True)

# copy best checkpoint
src = os.path.join(OUTPUT_DIR, 'checkpoints/best.pth')
dst = os.path.join(save_dir, 'fets_local_best.pth')
shutil.copy2(src, dst)
print(f'Saved to {dst}')
print(f'Size: {os.path.getsize(dst) / 1e6:.1f} MB')

---
## 12. What's Next

Once this notebook runs successfully:

1. **Check the Dice score** — target is ≥ 0.85 for clinical relevance
2. **Train longer / more data** — increase `MAX_TRAIN` and epochs if time allows
3. **Federated training** — run `train_federated.py` using all 3 datasets
4. **Deploy** — use the saved checkpoint with the FastAPI Docker container

To run federated training from Colab:
```python
!cd /content/FedFMS/fed_vis && python -m fedvis.scripts.train_federated \
    --rounds 20 --local_epochs 1 --data_root /content/data
```